In [1]:
import sys
import os

# Get the root directory of the project
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the root directory to the system path
sys.path.insert(0, project_root)

In [2]:
import pandas as pd 
import os
import json

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# import seaborn as sns
# sns.set_theme()

In [3]:
from explainable_dataset import *
from viz_utils import create_highlighted_passage
from IPython.display import display, HTML

def highlight_passage(tokens, binary_list):
    highlighted_passage = create_highlighted_passage(tokens,
                                                     binary_list,
                                                     None,
                                                     'green',
                                                     'linear')
    display(HTML("\n".join(highlighted_passage)))
    
tokenizer = AutoTokenizer.from_pretrained('meta-llama/Llama-3.1-70B-Instruct')


file_path = '../data/35_sample_dataset/35_random_samples_explained.jsonl'
dataset_me = ExplanationsDataset(file_path, tokenizer, decode_positive_as_list=True,
                                psg_key='positive', q_key='query'
                                )

file_path = '../data/35_sample_dataset/35_random_samples_llama3.1:70b-instruct-q8_0.jsonl'
dataset_llama = ExplanationsDataset(file_path, tokenizer, decode_positive_as_list=True  ,                                 
                                psg_key='positive', q_key='query')

# Iterate through the data
for i in range(len(dataset_llama)):
    print(dataset_me[i]["query"])
    
    print("Me")
    d = dataset_me[i]
    highlight_passage(d['tokenized_positive_decoded'], d['rationales'])
    
    print("Llma")  
    d = dataset_llama[i]
    highlight_passage(d['tokenized_positive_decoded'], d['rationales'])
    
    print("\n")

what is a system hardware
Me


Llma




meaning of the name lincoln
Me


Llma




do.muslims not believe the holocaust happened
Me


Llma


... (60 outputs left out)



what is principal company
Me


Llma




how do i move information from my old outlook emails to new computer
Me


Llma


In [4]:

dataset_paths = [
    '35_random_samples_explained.jsonl',
    '35_random_samples_gemma2:27b-instruct-q8_0.jsonl',
    '35_random_samples_gemma2:9b-instruct-q8_0.jsonl',
    '35_random_samples_gpt-4o-mini-2024-07-18.jsonl',
    '35_random_samples_gpt-4o-2024-08-06.jsonl',
    '35_random_samples_llama3.1:70b-instruct-q4_0.jsonl',
    '35_random_samples_llama3.1:70b-instruct-q8_0.jsonl',
    '35_random_samples_llama3.1:8b-instruct-fp16.jsonl',
    '35_random_samples_gemma2:27b-instruct-fp16.jsonl',
    '35_random_samples_gemma2:9b-instruct-fp16.jsonl'
]

datasets = {}
for dataset_path_r in dataset_paths:
    dataset_path = "../data/35_sample_dataset/" + dataset_path_r
    d = ExplanationsDataset(dataset_path, tokenizer, decode_positive_as_list=True,
                                psg_key='positive', q_key='query')
    dataset_name = dataset_path_r[len('35_random_samples_'):-len(".jsonl")]
    datasets[dataset_name] = d

datasets

{'explained': <explainable_dataset.ExplanationsDataset at 0x7f9b78a8c280>,
 'gemma2:27b-instruct-q8_0': <explainable_dataset.ExplanationsDataset at 0x7f9b78af4b50>,
 'gemma2:9b-instruct-q8_0': <explainable_dataset.ExplanationsDataset at 0x7f9ba0227430>,
 'gpt-4o-mini-2024-07-18': <explainable_dataset.ExplanationsDataset at 0x7f9b78af4df0>,
 'gpt-4o-2024-08-06': <explainable_dataset.ExplanationsDataset at 0x7f9b78af4a60>,
 'llama3.1:70b-instruct-q4_0': <explainable_dataset.ExplanationsDataset at 0x7f9b78af4b20>,
 'llama3.1:70b-instruct-q8_0': <explainable_dataset.ExplanationsDataset at 0x7f9b78af4f70>,
 'llama3.1:8b-instruct-fp16': <explainable_dataset.ExplanationsDataset at 0x7f9ad60b7fa0>,
 'gemma2:27b-instruct-fp16': <explainable_dataset.ExplanationsDataset at 0x7f9b78af4dc0>,
 'gemma2:9b-instruct-fp16': <explainable_dataset.ExplanationsDataset at 0x7f9b78af4940>}

In [5]:
def read_tf_idf_data():
    file_path = '../data/35_sample_dataset/35_random_samples_tf_idf.jsonl'
    with open(file_path, 'r') as f:
        data = [json.loads(line) for line in f]
    return data

tf_idf_data = read_tf_idf_data()
tf_idf_data

[{'query': 'what is a system hardware',
  'positive': 'Abbreviated as HW, Hardware is best described as any physical component of a computer system that contains a circuit board, ICs, or other electronics.A perfect example of hardware is the screen on which you are viewing this page.bbreviated as HW, Hardware is best described as any physical component of a computer system that contains a circuit board, ICs, or other electronics.',
  'negative': 'In essence, a computer is made up of software and hardware parts. Software refers to the programs that tell the hardware what to do. Software also comprises programs or applications accessed by the user. It is not possible to physically touch software, but computer users can see what software does on the screen. Hardware, on the other hand, are the physical components of a computer. The system unit is by far the most important and most delicate hardware part of a computer. It houses the processor, the storage components, and the computerâ\x80\

In [6]:
import pandas as pd

# Initialize the DataFrame to hold the data
columns = ['query']  # Start with 'query'
# Add columns dynamically based on the dataset keys
columns.extend([f"tokenized_positive_decoded_{key}" for key in datasets.keys()])
columns.extend([f"rationales_{key}" for key in datasets.keys()])

df = pd.DataFrame(columns=columns)

# Assuming all datasets have the same length
for i in range(len(next(iter(datasets.values())))):  # Use length of the first dataset
    row_data = {'query': datasets[next(iter(datasets))][i]['query']}  # Initialize with query
    
    # Iterate through each dataset in the dictionary
    for key, dataset in datasets.items():
        row_data[f"tokenized_positive_decoded_{key}"] = dataset[i]['tokenized_positive_decoded']
        row_data[f"rationales_{key}"] = dataset[i]['rationales']
    
    # Add tf-idf data
    row_data['rationales_tf_idf'] = torch.tensor(tf_idf_data[i]['score_mask'])
    row_data['rationales_select_all'] = torch.tensor(tf_idf_data[i]['select_all'])
    
    # Append the data to the DataFrame
    df = pd.concat([df, pd.DataFrame([row_data])], ignore_index=True)

    

# Output the DataFrame
df


,query,tokenized_positive_decoded_explained,tokenized_positive_decoded_gemma2:27b-instruct-q8_0,tokenized_positive_decoded_gemma2:9b-instruct-q8_0,tokenized_positive_decoded_gpt-4o-mini-2024-07-18,tokenized_positive_decoded_gpt-4o-2024-08-06,tokenized_positive_decoded_llama3.1:70b-instruct-q4_0,tokenized_positive_decoded_llama3.1:70b-instruct-q8_0,tokenized_positive_decoded_llama3.1:8b-instruct-fp16,tokenized_positive_decoded_gemma2:27b-instruct-fp16,...,rationales_gemma2:9b-instruct-q8_0,rationales_gpt-4o-mini-2024-07-18,rationales_gpt-4o-2024-08-06,rationales_llama3.1:70b-instruct-q4_0,rationales_llama3.1:70b-instruct-q8_0,rationales_llama3.1:8b-instruct-fp16,rationales_gemma2:27b-instruct-fp16,rationales_gemma2:9b-instruct-fp16,rationales_tf_idf,rationales_select_all
0,what is a system hardware,"[Abb, rev, iated, as, HW, ,, Hardware, is,...","[Abb, rev, iated, as, HW, ,, Hardware, is,...","[Abb, rev, iated, as, HW, ,, Hardware, is,...","[Abb, rev, iated, as, HW, ,, Hardware, is,...","[Abb, rev, iated, as, HW, ,, Hardware, is,...","[Abb, rev, iated, as, HW, ,, Hardware, is,...","[Abb, rev, iated, as, HW, ,, Hardware, is,...","[Abb, rev, iated, as, HW, ,, Hardware, is,...","[Abb, rev, iated, as, HW, ,, Hardware, is,...",...,"[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(True), tensor(True), tensor(True), ten..."
1,meaning of the name lincoln,"[American, Meaning, :, The, name, Lincoln,...","[American, Meaning, :, The, name, Lincoln,...","[American, Meaning, :, The, name, Lincoln,...","[American, Meaning, :, The, name, Lincoln,...","[American, Meaning, :, The, name, Lincoln,...","[American, Meaning, :, The, name, Lincoln,...","[American, Meaning, :, The, name, Lincoln,...","[American, Meaning, :, The, name, Lincoln,...","[American, Meaning, :, The, name, Lincoln,...",...,"[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(True), tensor(True), tensor(True), ten..."
2,do.muslims not believe the holocaust happened,"[For, the, majority, of, Muslims, in, th...","[For, the, majority, of, Muslims, in, th...","[For, the, majority, of, Muslims, in, th...","[For, the, majority, of, Muslims, in, th...","[For, the, majority, of, Muslims, in, th...","[For, the, majority, of, Muslims, in, th...","[For, the, majority, of, Muslims, in, th...","[For, the, majority, of, Muslims, in, th...","[For, the, majority, of, Muslims, in, th...",...,"[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(False), tensor(False), ...","[tensor(True), tensor(True), tensor(True), ten...","[tensor(True), tensor(True), tensor(True), ten...","[tensor(False), tensor(False), tensor(False), ...","[tensor(True), tensor(True), tensor(True), ten...","[tensor(False), tensor(False), tensor(False), ...","[tensor(False), tensor(True), tensor(False), t...","[tensor(True), tensor(True), tensor(True), ten..."
3,what illness do premature babies contract,"[Prem, ature, birth, can, lead, to, these...","[Prem, ature, birth, can, lead, to, these...","[Prem, ature, birth, can, lead, to, these...","[Prem, ature, birth, can, lead, to, these...","[Prem, ature, birth, can, lead

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd


def compute_stats(rationales_label):
    # Initialize variables to store the cumulative metrics
    precisions = []
    recalls = []
    f1_scores = []
    
    # Iterate through each row in the DataFrame
    for index, row in df.iterrows():
        # Get the rationales from both datasets
        rationales_llama = row["rationales_" + rationales_label]
        rationales_me = row['rationales_explained']

        y_true = rationales_me.tolist()
        y_pred = rationales_llama.tolist() 

        # Calculate precision, recall, and F1 score for this pair of rationales
        try:
            precision = precision_score(y_true, y_pred, zero_division=1)
            recall = recall_score(y_true, y_pred)
            f1 = f1_score(y_true, y_pred)
        except ValueError as e:
            print(f"{index}: {e}")
            print()
            continue
        # Append the scores to the lists
        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)

    # Calculate the average of each metric
    avg_precision = sum(precisions) / len(precisions)
    avg_recall = sum(recalls) / len(recalls)
    avg_f1 = sum(f1_scores) / len(f1_scores)

    return {
        'avg_f1': avg_f1,
        'avg_precision': avg_precision,
        'avg_recall': avg_recall,
        'f1_scores': f1_scores,
        'precisions': precisions,
        'recalls': recalls
    }

    
all_stats = {}
for model_name in datasets.keys():
    all_stats[model_name] = compute_stats(model_name)
    
all_stats['tf_idf'] = compute_stats('tf_idf')
all_stats['select_all'] = compute_stats('select_all')

all_stats = {k:v for k, v in sorted(all_stats.items(), key=lambda item: -item[1]['avg_f1'])}
for method, stats in all_stats.items():
    print(f"\n\tStats for {method}")
    print(f"F1 Score: {stats['avg_f1']:.4f}")
    print(f"Precision Score: {stats['avg_precision']:.4f}")
    print(f"Recall Score: {stats['avg_recall']:.4f}")

          
save_stats = True
if save_stats:
    with open("../data/other/f1_stats.json", "w") as file:
          json.dump(all_stats, file, indent=4)
          


	Stats for explained
F1 Score: 1.0000
Precision Score: 1.0000
Recall Score: 1.0000

	Stats for gemma2:27b-instruct-q8_0
F1 Score: 0.7045
Precision Score: 0.7592
Recall Score: 0.7093

	Stats for gemma2:27b-instruct-fp16
F1 Score: 0.7035
Precision Score: 0.7635
Recall Score: 0.7139

	Stats for gpt-4o-2024-08-06
F1 Score: 0.6456
Precision Score: 0.7667
Recall Score: 0.6197

	Stats for llama3.1:70b-instruct-q8_0
F1 Score: 0.6415
Precision Score: 0.7495
Recall Score: 0.6587

	Stats for llama3.1:70b-instruct-q4_0
F1 Score: 0.6376
Precision Score: 0.7587
Recall Score: 0.6371

	Stats for gpt-4o-mini-2024-07-18
F1 Score: 0.6202
Precision Score: 0.6823
Recall Score: 0.6959

	Stats for llama3.1:8b-instruct-fp16
F1 Score: 0.5446
Precision Score: 0.6263
Recall Score: 0.6042

	Stats for select_all
F1 Score: 0.4643
Precision Score: 0.3249
Recall Score: 1.0000

	Stats for gemma2:9b-instruct-fp16
F1 Score: 0.4449
Precision Score: 0.7574
Recall Score: 0.4037

	Stats for gemma2:9b-instruct-q8_0
F1 Score

In [8]:
# viz_datasets = [] # to see all
viz_datasets = ['gemma2:27b-instruct-q8_0', 'gemma2:9b-instruct-q8_0', 'explained']

viz_indexes = [] # to see all
# viz_indexes = [31, 26]

for i in range(len(dataset_llama)):
    if i not in viz_indexes and viz_indexes != []:
        continue
    
    print(f"{'X' * 20}  {datasets['explained'][i]['query']}  {'X' * 20}")
    for model_name, dataset in datasets.items():
        if model_name not in viz_datasets and viz_datasets != []:
            continue
            
        print(f"\t{model_name}")
        d = dataset[i]
        highlight_passage(d['tokenized_positive_decoded'], d['rationales'])

        print()
    print()
    print()
        

XXXXXXXXXXXXXXXXXXXX  what is a system hardware  XXXXXXXXXXXXXXXXXXXX
	explained



	gemma2:27b-instruct-q8_0



	gemma2:9b-instruct-q8_0





XXXXXXXXXXXXXXXXXXXX  meaning of the name lincoln  XXXXXXXXXXXXXXXXXXXX
	explained



	gemma2:27b-instruct-q8_0



	gemma2:9b-instruct-q8_0


... (95 outputs left out)


	gemma2:9b-instruct-q8_0





XXXXXXXXXXXXXXXXXXXX  how do i move information from my old outlook emails to new computer  XXXXXXXXXXXXXXXXXXXX
	explained



	gemma2:27b-instruct-q8_0



	gemma2:9b-instruct-q8_0


In [9]:
def get_selected_lens(explanations):
    return pd.Series(
    [
        len([x for x in xs if x == 1])
        for xs 
        in explanations
    ]
    )

all_rationales = [col_name for col_name in df.columns.values if col_name.startswith("rationales_")]
for rationale_name in all_rationales:

    print(f"\n\tLenghts of explanations {rationale_name} statistics:")
    explanations = df[rationale_name]
    print(get_selected_lens(explanations).describe())
    



	Lenghts of explanations rationales_explained statistics:
count     35.000000
mean      24.285714
std       19.283598
min        4.000000
25%       12.500000
50%       21.000000
75%       31.000000
max      105.000000
dtype: float64

	Lenghts of explanations rationales_gemma2:27b-instruct-q8_0 statistics:
count    35.000000
mean     21.542857
std      12.327260
min       2.000000
25%      13.500000
50%      19.000000
75%      29.000000
max      57.000000
dtype: float64

	Lenghts of explanations rationales_gemma2:9b-instruct-q8_0 statistics:
count    35.000000
mean     12.800000
std      10.585228
min       0.000000
25%       3.000000
50%      12.000000
75%      20.000000
max      36.000000
dtype: float64

	Lenghts of explanations rationales_gpt-4o-mini-2024-07-18 statistics:
count    35.000000
mean     25.914286
std      15.416514
min       2.000000
25%      15.500000
50%      24.000000
75%      37.000000
max      59.000000
dtype: float64

	Lenghts of explanations rationales_gpt-4o-20